# Phase 3 B1 — BrainSegFounder 4-Modality Re-Extraction (v2)


In [ ]:
# ── MUST BE FIRST CELL: install MONAI before any imports ─────────────────────
# Kaggle as of 2025-Q2 uses Python 3.12 where monai<1.3.2 breaks due to
# removal of importlib.find_module(). Force install compatible version.
import subprocess, sys
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     '--force-reinstall', '--no-deps', 'monai==1.4.0'],
    capture_output=True, text=True
)
if result.returncode == 0:
    print('MONAI 1.4.0 installed ✅')
else:
    # Fallback: try without --no-deps
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', '-q',
         '--force-reinstall', 'monai==1.4.0']
    )
    print('MONAI 1.4.0 installed (with deps) ✅')

# Clear any pre-loaded broken monai from module cache
for mod_name in list(sys.modules.keys()):
    if mod_name.startswith('monai'):
        del sys.modules[mod_name]
print('Module cache cleared — safe to import now')

# Phase 3 B1 — BSF Re-Extraction v2
## ROI Crop + Octant Spatial Pooling + Mask-Weighted Pooling

**Mirrors Phase2_B1** (MetSeg) but for BrainSegFounder (Swin UNETR).

---

### What This Notebook Does

| Fix | What is changed | Why |
|---|---|---|
| **ROI Crop** | Crop to WT bounding box + 8px padding, resize to 96³ | Single clean forward pass — full tumour always in view |
| **Octant Spatial Pooling** | Divide encoder10 feature map (6×6×6 @ 768ch) into 8 sub-cubes | Relative activation per octant encodes shape/direction |
| **Mask-Weighted Pooling** | Weight features by WT/TC/ET seg probabilities | Separates boundary (WT), core (TC), enhancement (ET) signals |

**Output:** 8×768 + 3×768 = **8448-dim** per scan (vs 768-dim GAP in v1)

**Runtime:** ~20–30 min on Kaggle T4 (inference only, no training)

---

### Kaggle Dataset Setup

Attach the following datasets before running:
1. **Cyprus PROTEAS MRI data** (same as used in A1B training)
2. **BSF fold checkpoints** — the 3 `bsf_fold{0,1,2}_best.pth` files from A1B
3. **data_splits.json** — patient fold assignment

```
Expected checkpoint paths (adjust CKPT_DIR below):
  /kaggle/input/{your-dataset}/bsf_fold0_best.pth
  /kaggle/input/{your-dataset}/bsf_fold1_best.pth
  /kaggle/input/{your-dataset}/bsf_fold2_best.pth
```

In [ ]:
# ── Imports ─────────────────────────────────────────────────────────────────
import os, sys, json, time, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
import nibabel as nib
from scipy.ndimage import zoom

# MONAI was installed in cell 0
from monai.networks.nets import SwinUNETR
import monai

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
print(f'MONAI  : {monai.__version__}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


In [ ]:
# ── Paths — same resolution logic as Phase3_A1B ─────────────────────────────
import glob as _glob
IS_KAGGLE = Path('/kaggle/input').exists()

if IS_KAGGLE:
    # Data root — same as A1B
    DATA_ROOT = Path('/kaggle/input/datasets/mohamedmohamed23/cyprus-proteas-brain-mets')
    if not DATA_ROOT.exists():
        for candidate in sorted(Path('/kaggle/input').rglob('data_splits.json')):
            DATA_ROOT = candidate.parent; break

    # Checkpoint search — find bsf_fold*_best.pth ANYWHERE in /kaggle/input
    # Mirrors A1B find_bsf_weights() approach
    _ckpt_hits = _glob.glob('/kaggle/input/**/bsf_fold*_best.pth', recursive=True)
    if not _ckpt_hits:
        # Broader: any large .pth file with 'fold' in the name
        _ckpt_hits = [f for f in _glob.glob('/kaggle/input/**/*.pth', recursive=True)
                      if 'fold' in f and os.path.getsize(f) > 100_000_000]
    if _ckpt_hits:
        CKPT_DIR = Path(_ckpt_hits[0]).parent
        print(f'Auto-found checkpoints in: {CKPT_DIR}')
    else:
        CKPT_DIR = Path('/kaggle/input/bsf-fold-checkpoints')
        print('WARNING: checkpoints not auto-found — update CKPT_DIR manually below')

    # ── MANUAL OVERRIDE — uncomment and set if auto-search fails ─────────────
    # CKPT_DIR = Path('/kaggle/input/YOUR-CHECKPOINT-DATASET/checkpoints')

    SPLITS_FILE = DATA_ROOT / 'data_splits.json'
    OUT_DIR     = Path('/kaggle/working/bsf_v2_embeddings')
    SYMLINK_DIR = Path('/kaggle/working/nifti_links')
else:
    ROOT        = Path('/home/moamed/canada_me/explainable_diseas/implementation_cyprus')
    DATA_ROOT   = ROOT / 'Data' / 'Cyprus-PROTEAS-zips'
    CKPT_DIR    = ROOT / 'Phase3' / 'bsf_fold_outputs' / 'checkpoints'
    SPLITS_FILE = DATA_ROOT / 'data_splits.json'
    OUT_DIR     = ROOT / 'Phase3' / 'bsf_fold_outputs' / 'embeddings_v2'
    SYMLINK_DIR = OUT_DIR / 'nifti_links'

OUT_DIR.mkdir(parents=True, exist_ok=True)
SYMLINK_DIR.mkdir(parents=True, exist_ok=True)

# ── resolve_path — identical to A1B ──────────────────────────────────────────
def resolve_path(root, rel):
    p = root / rel
    if p.exists(): return str(p)
    gz = str(p) + '.gz'
    if Path(gz).exists(): return gz
    nii_gz_alt = str(p).replace('.nii.gz', '.nii_gz')
    if Path(nii_gz_alt).exists():
        link = SYMLINK_DIR / rel
        link.parent.mkdir(parents=True, exist_ok=True)
        if not link.exists(): os.symlink(nii_gz_alt, str(link))
        return str(link)
    parent = p.parent
    if parent.exists():
        for f in parent.iterdir():
            if f.name.lower() == p.name.lower(): return str(f)
            nii_gz_name = str(f).replace('.nii_gz', '.nii.gz')
            if Path(nii_gz_name).name.lower() == p.name.lower():
                link = SYMLINK_DIR / rel
                link.parent.mkdir(parents=True, exist_ok=True)
                if not link.exists(): os.symlink(str(f), str(link))
                return str(link)
    raise FileNotFoundError(f'Cannot resolve: {rel}')

# ── Model config ─────────────────────────────────────────────────────────────
MODEL_CONFIG = dict(
    in_channels=4, out_channels=3, feature_size=48,
    use_checkpoint=False, spatial_dims=3,
    drop_rate=0.0, attn_drop_rate=0.0, dropout_path_rate=0.0,
)
TARGET_SIZE = (96, 96, 96)
ROI_PADDING = 8

# ── Print all .pth files found (debug) ───────────────────────────────────────
if IS_KAGGLE:
    all_pth = _glob.glob('/kaggle/input/**/*.pth', recursive=True)
    print(f'All .pth files in /kaggle/input ({len(all_pth)} found):')
    for f in all_pth:
        sz = os.path.getsize(f)/1e6
        print(f'  {f}  ({sz:.0f} MB)')

print('=== Path Check ===')
print(f'  DATA_ROOT : {DATA_ROOT}  exists={DATA_ROOT.exists()}')
print(f'  CKPT_DIR  : {CKPT_DIR}')
print(f'  SPLITS    : {SPLITS_FILE.exists()}')
for fold in range(3):
    ckpt = CKPT_DIR / f'bsf_fold{fold}_best.pth'
    ok   = '✅' if ckpt.exists() else '❌'
    sz   = f'({ckpt.stat().st_size/1e6:.0f} MB)' if ckpt.exists() else ''
    print(f'  {ok} bsf_fold{fold}_best.pth  {sz}')


In [ ]:
# ── Model factory + bottleneck hook ──────────────────────────────────────────
import inspect

def create_bsf_model():
    """Instantiate SwinUNETR with same config as A1B training."""
    kw = dict(**MODEL_CONFIG)
    sig = inspect.signature(SwinUNETR.__init__)
    if 'img_size' in sig.parameters:
        kw['img_size'] = TARGET_SIZE   # older MONAI versions need this
    return SwinUNETR(**kw)

def load_fold_checkpoint(model, fold):
    """Load fine-tuned BSF checkpoint for a specific fold."""
    ckpt_path = CKPT_DIR / f'bsf_fold{fold}_best.pth'
    sd = torch.load(str(ckpt_path), map_location='cpu', weights_only=False)
    # Handle various checkpoint formats
    if isinstance(sd, dict):
        for key in ('model_state_dict', 'state_dict', 'model'):
            if key in sd: sd = sd[key]; break
    missing, unexpected = model.load_state_dict(sd, strict=False)
    print(f'  Fold {fold}: loaded {ckpt_path.name}  '
          f'missing={len(missing)}  unexpected={len(unexpected)}')
    return model


# ── Bottleneck hook ───────────────────────────────────────────────────────────
# In MONAI SwinUNETR:
#   swinViT produces hidden_states_out (list of 5 feature maps)
#   encoder10 = final ConvBlock taking hidden_states_out[4] → bottleneck
#   For 96³ input + feature_size=48:
#     hidden_states_out[4] spatial = 96/(2*2*2*2) = 6 → 6×6×6 @ 768 channels
#     encoder10 output            = 6×6×6 @ 768 channels
#
# We hook encoder10 to capture the 6×6×6×768 feature map.

_bottleneck_cache = {}

def register_bottleneck_hook(model):
    """Hook encoder10 to cache bottleneck feature map."""
    _bottleneck_cache.clear()
    def _hook(module, inp, out):
        _bottleneck_cache['feat'] = out.detach()
    handle = model.encoder10.register_forward_hook(_hook)
    return handle

# Test model instantiation
print('Building SwinUNETR...')
_test_model = create_bsf_model()
print(f'Parameters: {sum(p.numel() for p in _test_model.parameters())/1e6:.1f}M')

# Check bottleneck shape with dummy input
_test_model.eval()
_handle = register_bottleneck_hook(_test_model)
with torch.no_grad():
    _dummy = torch.zeros(1, 4, 96, 96, 96)
    _ = _test_model(_dummy)
_handle.remove()
_bn_feat = _bottleneck_cache['feat']
print(f'Bottleneck feature map shape: {tuple(_bn_feat.shape)}')
# Expected: (1, 768, 6, 6, 6)
BN_CHANNELS = _bn_feat.shape[1]
BN_SPATIAL  = _bn_feat.shape[2:]   # (6, 6, 6)
del _test_model, _dummy, _bn_feat
torch.cuda.empty_cache() if torch.cuda.is_available() else None

EMB_DIM_OCTANT = 8 * BN_CHANNELS          # 8 × 768 = 6144
EMB_DIM_MASK   = 3 * BN_CHANNELS          # 3 × 768 = 2304
EMB_DIM_TOTAL  = EMB_DIM_OCTANT + EMB_DIM_MASK  # 8448
print(f'\nEmbedding dimensions:')
print(f'  Bottleneck: {BN_CHANNELS}ch @ {BN_SPATIAL}')
print(f'  Octant pool: 8 × {BN_CHANNELS} = {EMB_DIM_OCTANT}')
print(f'  Mask-weighted pool: 3 × {BN_CHANNELS} = {EMB_DIM_MASK}')
print(f'  Total v2 dim: {EMB_DIM_TOTAL}  (vs 768 for v1 GAP)')

In [ ]:
# ── Data loading — uses resolve_path (same symlink strategy as A1B) ──────────

def load_scan_from_dict(scan_dict, data_root):
    """
    Load 4-channel MRI + mask using paths from data_splits.json scan dict.
    Uses resolve_path() to handle .nii_gz Kaggle naming bug (same as A1B).
    """
    import nibabel as nib
    vols     = []
    vox_size = None

    for mod_key in ['t1', 't1c', 't2', 'fla']:
        rel = scan_dict.get(mod_key, '')
        try:
            fpath = resolve_path(data_root, rel)
            img   = nib.load(fpath)
            arr   = img.get_fdata(dtype=np.float32)
            if vox_size is None:
                vox_size = np.abs(np.diag(img.affine)[:3]).astype(np.float32)
            vols.append(arr)
        except FileNotFoundError:
            vols.append(None)

    if vols[3] is None:   # FLAIR required
        return None
    ref_shape = vols[3].shape
    vols      = [v if v is not None else np.zeros(ref_shape, np.float32) for v in vols]
    volume_np = np.stack(vols, axis=0)   # (4, D, H, W)

    # Load mask
    try:
        mask_path = resolve_path(data_root, scan_dict.get('mask', ''))
        seg_img   = nib.load(mask_path)
        seg_raw   = seg_img.get_fdata(dtype=np.float32)
        wt_mask   = (seg_raw > 0).astype(np.float32)
        if vox_size is None:
            vox_size = np.abs(np.diag(seg_img.affine)[:3]).astype(np.float32)
    except FileNotFoundError:
        return None

    return volume_np, wt_mask, seg_raw, vox_size


def z_score_normalise(volume):
    """Per-channel z-score normalisation (same as A1B training)."""
    out = volume.copy()
    for c in range(out.shape[0]):
        ch = out[c]
        out[c] = (ch - ch.mean()) / (ch.std() + 1e-8)
    return out


print('Data helpers defined ✅  (resolve_path symlink strategy active)')


In [ ]:
# ── ROI crop + resize helper ──────────────────────────────────────────────────
# Mirrors Phase2_B1 exactly but for 4-channel input and 96³ target size.

def roi_crop_resize(volume_np, wt_mask, target_size=(96, 96, 96), padding=8):
    """
    Crop to WT bounding box + padding, resize to target_size.

    Args:
        volume_np  : (4, D, H, W) float32
        wt_mask    : (D, H, W)   float32  (binary WT mask)
        target_size: (3,) output spatial size
        padding    : voxels added on each side

    Returns:
        cropped_vol  : (4, *target_size) float32
        cropped_mask : (1, *target_size) float32  (for mask-weighted pooling)
        cropped_seg  : (1, *target_size) float32  (raw labels, for TC/ET pooling)
        fallback     : True if mask was empty (used whole volume)
    """
    coords = np.argwhere(wt_mask > 0)
    fallback = len(coords) == 0

    D, H, W = wt_mask.shape
    if not fallback:
        z0, y0, x0 = coords.min(0)
        z1, y1, x1 = coords.max(0) + 1
        z0 = max(0, z0 - padding); z1 = min(D, z1 + padding)
        y0 = max(0, y0 - padding); y1 = min(H, y1 + padding)
        x0 = max(0, x0 - padding); x1 = min(W, x1 + padding)
    else:
        z0, y0, x0, z1, y1, x1 = 0, 0, 0, D, H, W

    # Crop
    vol_crop  = volume_np[:, z0:z1, y0:y1, x0:x1]   # (4, dz, dy, dx)
    mask_crop = wt_mask[z0:z1, y0:y1, x0:x1]        # (dz, dy, dx)

    # Compute zoom factors for each spatial dim
    factors = [t / s for t, s in zip(target_size, vol_crop.shape[1:])]

    # Resize each channel independently
    vol_resized  = np.stack([zoom(vol_crop[c], factors, order=1) for c in range(vol_crop.shape[0])])
    mask_resized = zoom(mask_crop, factors, order=0)   # nearest for mask
    mask_resized = (mask_resized > 0.5).astype(np.float32)

    return vol_resized, mask_resized[np.newaxis], fallback


def build_subregion_masks_from_seg(seg_raw_np, target_size=(96, 96, 96)):
    """
    Build WT / TC / ET binary masks from raw seg labels (Cyprus-PROTEAS convention).
    Cyprus-PROTEAS label convention (matches A1B training):
        1 = NCR (necrotic core) → part of TC
        2 = ED  (edema)         → WT only
        3 = ET  (enhancing)     → part of TC and ET
    WT = labels {1,2,3}, TC = labels {1,3}, ET = labels {3}
    """
    factors = [t / s for t, s in zip(target_size, seg_raw_np.shape)]
    seg_r   = zoom(seg_raw_np, factors, order=0)
    wt  = (seg_r > 0).astype(np.float32)
    tc  = ((seg_r == 1) | (seg_r == 3)).astype(np.float32)
    et  = (seg_r == 3).astype(np.float32)
    return wt, tc, et


print('ROI crop helpers defined ✅')

In [ ]:
# ── Octant + Mask-weighted pooling ────────────────────────────────────────────

def octant_pool(feat_map):
    """
    Split spatial feature map into 8 octants and average-pool each.

    Args:
        feat_map: (1, C, D, H, W) tensor — bottleneck output

    Returns:
        (8*C,) numpy array — 8 octant vectors concatenated
    """
    _, C, D, H, W = feat_map.shape
    dh, hh, wh = D // 2, H // 2, W // 2
    # Handle odd spatial dims
    d_slices = [slice(0, dh), slice(dh, D)]
    h_slices = [slice(0, hh), slice(hh, H)]
    w_slices = [slice(0, wh), slice(wh, W)]

    octants = []
    for ds in d_slices:
        for hs in h_slices:
            for ws in w_slices:
                region = feat_map[:, :, ds, hs, ws]     # (1, C, d', h', w')
                pooled = F.adaptive_avg_pool3d(region, 1).flatten()  # (C,)
                octants.append(pooled)

    return torch.cat(octants).cpu().numpy()  # (8*C,)


def mask_weighted_pool(feat_map, wt_mask_t, tc_mask_t, et_mask_t):
    """
    Weighted average of bottleneck features by subregion masks.

    Args:
        feat_map  : (1, C, D, H, W) — bottleneck output
        wt_mask_t : (1, 1, d, h, w) — whole tumour binary mask (at TARGET_SIZE)
        tc_mask_t : (1, 1, d, h, w) — tumour core binary mask
        et_mask_t : (1, 1, d, h, w) — enhancing tumour binary mask

    Returns:
        (3*C,) numpy array — [wt_vec, tc_vec, et_vec] concatenated
    """
    C, D, H, W = feat_map.shape[1], *feat_map.shape[2:]
    # Downscale masks to match feature map spatial size
    def _down(m): return F.interpolate(
        m.float(), size=(D, H, W), mode='nearest').squeeze(0)  # (1, D, H, W)

    vecs = []
    for mask_t in [wt_mask_t, tc_mask_t, et_mask_t]:
        m = _down(mask_t)                              # (1, D, H, W)
        denom = m.sum() + 1e-6
        wvec = (feat_map.squeeze(0) * m).sum(dim=[1, 2, 3]) / denom  # (C,)
        vecs.append(wvec)

    return torch.cat(vecs).cpu().numpy()  # (3*C,)


print('Pooling functions defined ✅')
print(f'Each embedding will be: {EMB_DIM_TOTAL}-dim  '
      f'({EMB_DIM_OCTANT} octant + {EMB_DIM_MASK} mask-weighted)')

In [ ]:
# ── Single-scan embedding extractor ─────────────────────────────────────────

@torch.no_grad()
def extract_embedding_v2(model, scan_key, data_root):
    """
    ROI crop -> single forward pass -> octant + mask pooling.
    Returns: (EMB_DIM_TOTAL,) float32 array, or None if data not found.
    """
    scan_dict = SCAN_DICT_MAP.get(scan_key)
    if scan_dict is None:
        return None

    # 1. Load data using exact paths from splits JSON
    result = load_scan_from_dict(scan_dict, data_root)
    if result is None:
        return None
    volume_np, wt_mask, seg_raw, vox_size = result

    # 2. z-score normalise
    volume_np = z_score_normalise(volume_np)

    # 3. ROI crop to WT bounding box -> resize to 96^3
    vol_crop, wt_crop, fallback = roi_crop_resize(
        volume_np, wt_mask, target_size=TARGET_SIZE, padding=ROI_PADDING)

    # 4. Build subregion masks at TARGET_SIZE
    wt_t, tc_t, et_t = build_subregion_masks_from_seg(seg_raw, target_size=TARGET_SIZE)

    # 5. To tensor
    x         = torch.from_numpy(vol_crop).unsqueeze(0).to(device)   # (1,4,96,96,96)
    wt_mask_t = torch.from_numpy(wt_t).unsqueeze(0).unsqueeze(0).to(device)
    tc_mask_t = torch.from_numpy(tc_t).unsqueeze(0).unsqueeze(0).to(device)
    et_mask_t = torch.from_numpy(et_t).unsqueeze(0).unsqueeze(0).to(device)

    # 6. Forward pass (bottleneck captured by hook)
    _bottleneck_cache.clear()
    _ = model(x)
    feat = _bottleneck_cache.get('feat')
    if feat is None:
        return None

    # 7. Octant spatial pooling -> (8*C,)
    oct_vec  = octant_pool(feat)

    # 8. Mask-weighted pooling -> (3*C,)
    mask_vec = mask_weighted_pool(feat, wt_mask_t, tc_mask_t, et_mask_t)

    # 9. Concatenate -> (8448,)
    return np.concatenate([oct_vec, mask_vec]).astype(np.float32)


print('Extractor function defined ✅')


In [ ]:
# ── Build scan index from data_splits.json (3fold format) ───────────────────
# Format: splits['3fold']['fold_0']['train_scans'] + ['test_scans']
# Each scan = dict with patient_dir, visit, t1, t1c, t2, fla, mask paths

with open(SPLITS_FILE) as f:
    splits = json.load(f)

folds_data = splits['3fold']   # fold_0, fold_1, fold_2

# Collect ALL unique scans across all folds (train + test)
# scan_key = '{patient_dir}__{visit}'
SCAN_DICT_MAP = {}   # scan_key -> scan dict with file paths
for fold_name, fold_info in folds_data.items():
    for split_key in ['train_scans', 'test_scans']:
        for item in fold_info.get(split_key, []):
            key = f"{item['patient_dir']}__{item['visit']}"
            SCAN_DICT_MAP[key] = item

SCAN_INDEX = sorted(SCAN_DICT_MAP.keys())

print(f'Total unique scans (all folds combined): {len(SCAN_INDEX)}')
print('Sample scan keys:')
for k in SCAN_INDEX[:5]:
    item = SCAN_DICT_MAP[k]
    print(f'  {k}  fla={item["fla"]}')

# Map fold name → set of scan keys (for fold-aware naming if needed)
FOLD_SCAN_KEYS = {}
for fold_name, fold_info in folds_data.items():
    fold_id = int(fold_name.split('_')[1])   # 'fold_0' -> 0
    fold_keys = set()
    for split_key in ['train_scans', 'test_scans']:
        for item in fold_info.get(split_key, []):
            fold_keys.add(f"{item['patient_dir']}__{item['visit']}")
    FOLD_SCAN_KEYS[fold_id] = fold_keys
    print(f'  Fold {fold_id}: {len(fold_keys)} scans  '
          f'(train={fold_info["n_train_scans"]}, test={fold_info["n_test_scans"]})')


In [ ]:
# ── Per-fold extraction loop ─────────────────────────────────────────────────
#
# Run all 3 fold models on ALL scans, then average per scan.
# Matches the strategy used in Phase3_A4 (v1 embeddings).

all_fold_embs = []   # list of dicts: {scan_key: (8448,) array}
fold_metas    = []

t_total = time.time()

for fold in range(3):
    ckpt_path = CKPT_DIR / f'bsf_fold{fold}_best.pth'
    if not ckpt_path.exists():
        print(f'WARNING: Fold {fold} checkpoint not found — skipping')
        continue

    print(f'\n{"="*60}')
    print(f'  FOLD {fold} — loading checkpoint')
    print(f'{"="*60}')

    model = create_bsf_model().to(device)
    model = load_fold_checkpoint(model, fold)
    model.eval()
    hook_handle = register_bottleneck_hook(model)

    fold_embs = {}
    n_ok = n_skip = 0
    t_fold = time.time()

    for i, scan_key in enumerate(SCAN_INDEX):
        try:
            emb = extract_embedding_v2(model, scan_key, DATA_ROOT)
            if emb is not None:
                fold_embs[scan_key] = emb
                n_ok += 1
            else:
                n_skip += 1
        except Exception as e:
            print(f'  WARNING {scan_key}: {e}')
            n_skip += 1

        if (i + 1) % 20 == 0 or (i + 1) == len(SCAN_INDEX):
            elapsed = time.time() - t_fold
            eta     = elapsed / (i + 1) * (len(SCAN_INDEX) - i - 1)
            print(f'  [{i+1:3d}/{len(SCAN_INDEX)}] ok={n_ok} skip={n_skip}  '
                  f'elapsed={elapsed:.0f}s  ETA={eta:.0f}s')

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    hook_handle.remove()

    # Save fold npz
    fold_out = OUT_DIR / f'bsf_embeddings_fold{fold}_v2.npz'
    np.savez_compressed(str(fold_out), **fold_embs)

    meta = {
        'fold': fold, 'n_scans': n_ok, 'n_skipped': n_skip,
        'emb_dim': EMB_DIM_TOTAL,
        'emb_dim_octant': EMB_DIM_OCTANT,
        'emb_dim_mask': EMB_DIM_MASK,
        'bn_channels': BN_CHANNELS,
        'bn_spatial': list(BN_SPATIAL),
        'target_size': list(TARGET_SIZE),
        'roi_padding': ROI_PADDING,
        'model': 'SwinUNETR (BSF fine-tuned)',
        'pooling': 'octant_8x + mask_weighted_3x',
        'elapsed_s': round(time.time() - t_fold, 1),
        'keys': sorted(fold_embs.keys()),
    }
    with open(str(fold_out).replace('.npz', '_meta.json'), 'w') as fh:
        json.dump(meta, fh, indent=2)

    size_mb = fold_out.stat().st_size / 1e6
    print(f'\n  Fold {fold} done: {n_ok} scans @ {EMB_DIM_TOTAL}-dim  '
          f'-> {fold_out.name}  ({size_mb:.1f} MB)')

    all_fold_embs.append(fold_embs)
    fold_metas.append(meta)

    del model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

print(f'\nAll folds extracted in {(time.time()-t_total)/60:.1f} min')


In [ ]:
# ── Average across folds → consensus embedding per scan ─────────────────────

if not all_fold_embs:
    raise RuntimeError(
        'No fold embeddings were extracted!\n'
        'The checkpoint files were not found. Check the output above:\n'
        '  - Look at the .pth file list printed in cell_config\n'
        '  - Set CKPT_DIR manually in cell_config to the correct path\n'
        '  - Make sure the dataset containing bsf_fold*_best.pth is attached'
    )

all_keys = sorted(set().union(*[d.keys() for d in all_fold_embs]))
print(f'Keys across all folds: {len(all_keys)}')

averaged = {}
for key in all_keys:
    vecs = [d[key] for d in all_fold_embs if key in d]
    averaged[key] = np.mean(vecs, axis=0).astype(np.float32)

n_scans = len(averaged)
emb_dim = list(averaged.values())[0].shape[0] if averaged else 0
print(f'Averaged embeddings: {n_scans} scans x {emb_dim}-dim')

# Save averaged npz
avg_out = OUT_DIR / 'bsf_embeddings_averaged_v2.npz'
np.savez_compressed(str(avg_out), **averaged)
print(f'Saved: {avg_out.name}  ({avg_out.stat().st_size/1e6:.1f} MB)')

print('\nSample entries:')
for k in list(averaged.keys())[:4]:
    v = averaged[k]
    print(f'  {k}: shape={v.shape}  mean={v.mean():.4f}  std={v.std():.4f}')

X_all = np.stack(list(averaged.values()))
print(f'\nHas NaN: {np.isnan(X_all).any()}  Has Inf: {np.isinf(X_all).any()}')
print(f'Value range: [{X_all.min():.4f} .. {X_all.max():.4f}]')


In [ ]:
# ── Quick geometric probe: Volume R2 sanity check ────────────────────────────
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import r2_score
import nibabel as nib

# Compute volumes from masks using paths already in SCAN_DICT_MAP
vols = {}
for scan_key, item in SCAN_DICT_MAP.items():
    mask_path = DATA_ROOT / item['mask']
    if not mask_path.exists(): continue
    try:
        seg_img  = nib.load(str(mask_path))
        seg_data = seg_img.get_fdata()
        vox_size = np.abs(np.diag(seg_img.affine)[:3])
        vol = float((seg_data > 0).sum()) * float(np.prod(vox_size))
        vols[scan_key] = vol
    except: pass

# Align with averaged embeddings
common = sorted(set(averaged) & set(vols))
print(f'Common (embedding n volumes): {len(common)}')

if len(common) >= 10:
    X = np.array([averaged[k] for k in common])
    y = np.log1p([vols[k] for k in common])

    sc  = StandardScaler()
    pca = PCA(n_components=min(30, X.shape[0]-1, X.shape[1]))
    X_r = pca.fit_transform(sc.fit_transform(X))

    from sklearn.model_selection import LeaveOneOut
    preds = np.zeros(len(common))
    for tr, te in LeaveOneOut().split(X_r):
        ridge = Ridge(alpha=1.0).fit(X_r[tr], np.array(y)[tr])
        preds[te] = ridge.predict(X_r[te])

    r2_vol = r2_score(y, preds)
    flag = 'PASS' if r2_vol >= 0.60 else ('CLOSE' if r2_vol >= 0.4 else 'FAIL')
    print(f'Quick probe M1 Volume R2 (LOO-CV): {r2_vol:.3f}  {flag}  (threshold=0.60)')
    print(f'PCA variance explained: {pca.explained_variance_ratio_.sum():.2%}')
    print(f'Note: v1 GAP BSF was R2=0.017')
else:
    print('Not enough scans for probe check')


In [ ]:
# ── Final summary + download instructions ─────────────────────────────────────

print('='*65)
print('  BSF v2 Re-Extraction Complete')
print('='*65)
print(f'  Embedding dim : {EMB_DIM_TOTAL} (octant {EMB_DIM_OCTANT} + mask {EMB_DIM_MASK})')
print(f'  Total scans   : {len(averaged)}')
print(f'  v1 dim (GAP)  : {BN_CHANNELS}')
print(f'  Improvement   : {EMB_DIM_TOTAL/BN_CHANNELS:.0f}× richer embedding')
print()
print('  Files saved to:', OUT_DIR)
for fold in range(3):
    f = OUT_DIR / f'bsf_embeddings_fold{fold}_v2.npz'
    if f.exists():
        print(f'    bsf_embeddings_fold{fold}_v2.npz  ({f.stat().st_size/1e6:.1f} MB)')
if avg_out.exists():
    print(f'    bsf_embeddings_averaged_v2.npz  ({avg_out.stat().st_size/1e6:.1f} MB)')

print()
print('  Download and copy to:')
print('    implementation_cyprus/Phase3/bsf_fold_outputs/embeddings_v2/')
print()
print('  Next Step: Run Phase3_A4B_HybridBSF_Eval.ipynb')
print('    - Set BSF_EMB_DIR to embeddings_v2/')
print('    - Expected M1 Volume R² > 0.60  (vs 0.017 for pure BSF v1)')
print('='*65)